In [ ]:
import os
import re
import csv
import statistics

def parse_sol_file(filepath):
    """
    解析 _sol.txt 文件，返回 Cost 和 Time 的列表
    """
    if not os.path.exists(filepath):
        return [], []

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
        # 匹配行首的 Cost (排除 DetCost 等)
        costs_str = re.findall(r'^Cost\s+([\d\.]+)', content, re.MULTILINE)
        # 匹配行首的 Time
        times_str = re.findall(r'^Time\s+([\d\.]+)', content, re.MULTILINE)
        
        costs = [float(c) for c in costs_str]
        times = [float(t) for t in times_str]
        
    return costs, times

def parse_sim_avg(filepath):
    """
    解析 _solSim.txt 文件，提取每次运行对应的 Avg 值
    """
    if not os.path.exists(filepath):
        return []

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
        # 匹配以 Avg 开头的行，代表该解决方案在测试场景下的平均目标值
        avgs_str = re.findall(r'^Avg\s+([\d\.]+)', content, re.MULTILINE)
        avgs = [float(a) for a in avgs_str]
        
    return avgs

def main():
    # --- 配置区域 ---
    result_folder = 'UncertaintyDistribution/SimHGSHigh'   # 结果所在的文件夹名称
    instance_list_file = 'instance.txt' # 算例列表文件（假设在当前目录）
    output_csv = 'result_summary_SCDSPHigh.csv'   # 输出文件名
    target_count = 10          # 统计最后多少次
    # ----------------

    results = []

    # 检查算例列表文件是否存在
    if not os.path.exists(instance_list_file):
        print(f"错误: 当前目录下找不到文件 {instance_list_file}")
        return

    # 检查结果文件夹是否存在
    if not os.path.exists(result_folder):
        print(f"错误: 找不到结果文件夹 '{result_folder}'")
        return

    with open(instance_list_file, 'r', encoding='utf-8') as f:
        instances = [line.strip() for line in f if line.strip()]

    print(f"开始处理 {len(instances)} 个算例 (数据文件夹: {result_folder})...")

    for inst in instances:
        # 构建路径：NOFAST/算例名_sol.txt
        sol_path = os.path.join(result_folder, f"{inst}_sol.txt")
        sim_path = os.path.join(result_folder, f"{inst}_solSim.txt")
        
        # 1. 提取数据
        costs, times = parse_sol_file(sol_path)
        sim_avgs = parse_sim_avg(sim_path)
        
        # 2. 数据检查与对齐
        if not costs:
            print(f"警告: {inst} 无 Cost 数据 (文件路径: {sol_path})，跳过。")
            continue

        # 取三个列表长度的最小值，确保 Cost, Time, SimAvg 一一对应
        # 如果 SimAvg 文件还没生成，长度可能为0，此时 sim_avgs 为空
        min_len = min(len(costs), len(times))
        if len(sim_avgs) > 0:
            min_len = min(min_len, len(sim_avgs))
        
        # 截取有效部分
        valid_costs = costs[-min_len:]
        valid_times = times[-min_len:]
        valid_sim_avgs = sim_avgs[-min_len:] if sim_avgs else []

        # 3. 截取最后 N 次 (比如最后10次)
        last_n_costs = valid_costs[-target_count:]
        last_n_times = valid_times[-target_count:]
        last_n_sims = valid_sim_avgs[-target_count:]

        try:
            if not last_n_costs:
                raise ValueError("数据不足或对齐后为空")

            # A. 最好 Cost (最小)
            best_cost = min(last_n_costs)
            
            # B. 平均 Cost
            avg_cost = statistics.mean(last_n_costs)
            
            # C. 平均 Time
            avg_time = statistics.mean(last_n_times)
            
            # D. 最好 Cost 对应的 测试场景Avg
            # 找到 best_cost 在列表中的索引
            best_idx = last_n_costs.index(best_cost)
            
            # 取出对应的 Sim Avg
            if last_n_sims and best_idx < len(last_n_sims):
                target_sim_avg = last_n_sims[best_idx]
            else:
                target_sim_avg = "N/A" # 如果Sim文件缺失或行数不对齐

            # 存入结果
            results.append([
                inst, 
                best_cost, 
                f"{avg_cost:.2f}", 
                f"{avg_time:.2f}", 
                target_sim_avg
            ])
            
        except Exception as e:
            print(f"处理算例 {inst} 时发生错误: {e}")

    # 4. 写入 CSV
    header = ['算例名称', '最好Cost', '平均Cost', '平均时间', '测试场景Avg(对应最好解)']
    
    with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(results)

    print(f"\n处理完成！结果已保存至 {output_csv}")

if __name__ == "__main__":
    main()

开始处理 72 个算例 (数据文件夹: Distribution/High)...

处理完成！结果已保存至 result_summary_SCDSPHigh.csv


In [ ]:
import os
import csv

def parse_aff_file(filepath):
    """
    读取 AFF 文件夹下的文件，提取其中的单个数字 (AffScenarios)
    """
    if not os.path.exists(filepath):
        return None

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            if content:
                # 尝试转换为浮点数或整数
                return float(content)
    except Exception as e:
        print(f"读取文件 {filepath} 出错: {e}")
    
    return None

def main():
    # --- 配置区域 ---
    result_folder = 'AffScenarios'             # AFF 文件夹
    instance_list_file = 'instance.txt' # 算例列表文件
    output_csv = 'result_aff.csv'     # 输出文件名
    # ----------------

    results = []

    # 检查基础文件和文件夹
    if not os.path.exists(instance_list_file):
        print(f"错误: 当前目录下找不到文件 {instance_list_file}")
        return

    if not os.path.exists(result_folder):
        print(f"错误: 找不到文件夹 '{result_folder}'")
        return

    # 读取算例名
    with open(instance_list_file, 'r', encoding='utf-8') as f:
        instances = [line.strip() for line in f if line.strip()]

    print(f"开始处理 {len(instances)} 个算例 (目标文件夹: {result_folder})...")

    for inst in instances:
        # 假设 AFF 文件夹下的文件名是 算例名.txt 
        # 如果你的文件名有后缀（比如 inst_aff.txt），请修改下面这一行
        file_path = os.path.join(result_folder, f"{inst}_sol_Aff.txt")
        
        # 提取数字
        aff_value = parse_aff_file(file_path)
        
        if aff_value is not None:
            results.append([inst, aff_value])
        else:
            print(f"警告: 无法从 {file_path} 获取数据，跳过。")
            results.append([inst, "N/A"])

    # --- 写入 CSV ---
    header = ['算例名称', 'AffScenarios']
    
    try:
        with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(results)
        print(f"\n处理完成！结果已保存至 {output_csv}")
    except PermissionError:
        print(f"\n错误: 无法写入 {output_csv}，请检查该文件是否被 Excel 占用。")

if __name__ == "__main__":
    main()

开始处理 72 个算例 (目标文件夹: AFF)...

处理完成！结果已保存至 result_aff.csv
